In [19]:
from QASMBench.interface import qiskit

NAM_25 = [
    "adder_n24", "barenco_tof_3_n5", "barenco_tof_4_n7", "barenco_tof_5_n9", 
    "barenco_tof_10_n19", "csla_mux_3_original_n15", "csum_mux_9_corrected_n30", 
    "gf2^4_mult_n12", "gf2^5_mult_n15", "gf2^6_mult_n18", "gf2^7_mult_n21", 
    "gf2^8_mult_n24", "gf2^9_mult_n27", "mod5_4_n5", "mod_mult_55_n9", 
    "mod_red_21_n11", "qcla_adder_10_n36", "qcla_com_7_n24", "qcla_mod_7_n26", 
    "rc_adder_6_n14", "tof_3_n5", "tof_4_n7", "tof_5_n9", "tof_10_n19", 
    "vbe_adder_3_n10"
]

# load across both categories since Nam 25 spans small and medium
bm_small = qiskit.QASMBenchmark("QASMBench", "small", remove_final_measurements=True, do_transpile=False)
bm_medium = qiskit.QASMBenchmark("QASMBench", "medium", remove_final_measurements=True, do_transpile=False)

# check which names actually exist first
all_names = set(bm_small.circ_name_list + bm_medium.circ_name_list)
print(all_names)

{'bigadder_n18', 'qaoa_n3', 'teleportation_n3', 'iswap_n2', 'error_correctiond3_n5', 'cat_state_n22', 'variational_n4', 'wstate_n3', 'sat_n11', 'vqe_uccsd_n8', 'factor247_n15', 'dnn_n2', 'pea_n5', 'bv_n19', 'qram_n20', 'vqe_uccsd_n4', 'adder_n4', 'ipea_n2', 'qrng_n4', 'qec_en_n5', 'shor_n5', 'square_root_n18', 'sat_n7', 'hhl_n7', 'adder_n10', 'hhl_n14', 'qpe_n9', 'bell_n4', 'ising_n10', 'qaoa_n6', 'wstate_n27', 'vqe_uccsd_n6', 'cc_n12', 'quantumwalks_n2', 'dnn_n16', 'linearsolver_n3', 'qec9xz_n17', 'bwt_n21', 'vqe_n4', 'qec_sm_n5', 'simon_n6', 'qft_n4', 'dnn_n8', 'bv_n14', 'ising_n26', 'multiply_n13', 'hs4_n4', 'toffoli_n3', 'lpn_n5', 'seca_n11', 'multiplier_n15', 'basis_change_n3', 'basis_trotter_n4', 'gcm_n13', 'deutsch_n2', 'bb84_n8', 'qf21_n15', 'ghz_state_n23', 'vqe_n24', 'knn_n25', 'swap_test_n25', 'inverseqft_n4', 'cat_state_n4', 'fredkin_n3', 'grover_n2', 'qft_n18', 'hhl_n10'}


In [29]:
%load_ext autoreload
%autoreload 2

from quantum_optimiser import integration
from qiskit import transpile
import pyzx 
circuit = bm[1]
print(circuit)
diagram = integration.qiskit_to_pyzx(circuit)
pyzx.draw(diagram, labels="true")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
     ┌─────────┐┌─────────┐┌─────────┐┌────────────┐┌───────────────┐ »
q_0: ┤ Rx(1.1) ├┤ Ry(1.1) ├┤ Rz(1.1) ├┤ Rz(11π/10) ├┤ U3(π/2,0,π/4) ├─»
     ├─────────┤├─────────┤├─────────┤├────────────┤├───────────────┴┐»
q_1: ┤ Rx(1.1) ├┤ Ry(1.1) ├┤ Rz(1.1) ├┤ Rz(11π/10) ├┤ U3(π/2,π,3π/4) ├»
     └─────────┘└─────────┘└─────────┘└────────────┘└────────────────┘»
«     ┌─────────┐     ┌──────────┐┌───┐                            »
«q_0: ┤ Rx(π/2) ├──■──┤ Rx(2π/5) ├┤ X ├─────────────────────────■──»
«     └─────────┘┌─┴─┐├─────────┬┘└─┬─┘┌──────────┐┌─────────┐┌─┴─┐»
«q_1: ───────────┤ X ├┤ Ry(π/2) ├───■──┤ Rx(-π/2) ├┤ Rz(π/2) ├┤ X ├»
«                └───┘└─────────┘      └──────────┘└─────────┘└───┘»
«      ┌─────────────────┐ ┌─────────────┐┌─────────┐     ┌──────────┐┌───┐»
«q_0: ─┤ U3(π/2,2.042,π) ├─┤ U3(0,π,π/2) ├┤ Rx(π/2) ├──■──┤ Rx(2π/5) ├┤ X ├»
«     ┌┴─────────────────┴┐├─────────────┤└─────────┘

In [ ]:
from quantum_optimiser.multimetric import loss, simulated_annealing

print("Initial stats:", loss._compute_stats(circuit))

optimised, diagram, _, history = simulated_annealing.simulated_annealing_zx(
    circuit=circuit,
    cost_function=loss.log_weighted_loss,
    get_neighbor=simulated_annealing.get_neighbor_weighted_rules,
    initial_temp=100.0,
    cooling_rate=0.99,
    max_iterations=500,
    max_no_improvement=50,
    hardware=None
)

print("Final stats:", loss._compute_stats(optimised))
qiskit_optimised = transpile(circuit, basis_gates=["cx", "h", "t", "tdg", "s", "sdg", "x", "y", "z", "rz"], optimization_level=3)
print("Qiskit level 3:", loss._compute_stats(qiskit_optimised))


Initial stats: {'q': 2, 'g2': 42, 'g': 226, 't': 0, 'd': 154}
